In [5]:
import json
import base64
import numpy as np
import random
import networkx as nx

import plotly.graph_objects as go
import ipywidgets as widgets

from IPython.display import HTML, display
from networkx.readwrite import json_graph

def arg_vec(u, v):
    rcos = np.sum(u * v)
    rsin = u[0] * v[1] - u[1] * v[0]
    return np.arctan2(rsin, rcos)


def minimum_image_displacement(obj, p1, p2):
    disp = p2 - p1

    disp9 = np.vstack([
        disp + shift
        for shift in obj.shifts
    ])

    dist_sq9 = np.sum(disp9 ** 2, axis=1)

    k = np.argmin(dist_sq9)

    return disp9[k], k

def build_base_graph(obj):

    normal_x = []
    normal_y = []

    periodic_x = []
    periodic_y = []

    obj.edge_geometry = []

    for edge_id, (i, j) in enumerate(obj.edges):
        p1 = obj.nodes[i]
        p2 = obj.nodes[j]

        disp, k = minimum_image_displacement(obj, p1, p2)

        p2_image = p1 + disp

        x = [p1[0], p2_image[0]]
        y = [p1[1], p2_image[1]]

        obj.edge_geometry.append({
            "edge_id": edge_id,
            "edge": (i, j),
            "p1": p1.copy(),
            "p2": p2_image.copy(),
            "periodic": (k != 4)
        })

        if k == 4:
            normal_x.extend(x + [np.nan])
            normal_y.extend(y + [np.nan])
        else:
            periodic_x.extend(x + [np.nan])
            periodic_y.extend(y + [np.nan])

    return (
        np.array(normal_x),
        np.array(normal_y),
        np.array(periodic_x),
        np.array(periodic_y)
    )

def graph_to_json(obj):
    data = {
        "L": obj.L,
        "fig_frame": obj.fig_frame,
        "nodes": obj.nodes.tolist(),
        "edges": [
            [int(u), int(v)] for u, v in obj.edges
        ],
        "initial_edges": [
            [int(u), int(v)] for u, v in obj.initial_edges
        ],
        "move_history": obj.move_history,
        "current_step": int(obj.current_step)
    }

    return json.dumps(
        data,
        ensure_ascii=False,
        allow_nan=False
    )
    
def display_message(message_widget, message, best_score=None, best_score_step=None):
    text = message.replace("\n", "<br>")
    
    if best_score is not None and best_score_step is not None:
        text += (
            f"<br>Best score so far: "
            f"{best_score} at step {best_score_step}"
        )
    
    message_widget.value = (
        "<div style='color: gray;'>"
        f"{text}"
        "</div>"
    )

def update_history_controls(obj, step_spinbox, max_step_label):
    max_step = len(obj.move_history)

    step_spinbox.max = max_step
    step_spinbox.value = obj.current_step

    max_step_label.value = f"/{max_step}"

def update_score(score_widget, score):
    score_widget.value = (
        f"<div style='color: rgb(90, 38, 32); "
        f"background-color: rgb(248, 246, 235); "
        f"font-size: 36px; font-weight: 500; "
        f"text-align: center; padding: 6px 12px;'>"
        f"Score: {score:,.0f}"
        f"</div>"
    )
def update_view(obj, fig, traces):
    # 表示範囲を取得
    x_range = fig.layout.xaxis.range
    y_range = fig.layout.yaxis.range

    xmin, xmax = x_range
    ymin, ymax = y_range

    ix_min = int(np.floor(xmin / obj.L)) - 1
    ix_max = int(np.floor(xmax / obj.L)) + 1

    iy_min = int(np.floor(ymin / obj.L)) - 1
    iy_max = int(np.floor(ymax / obj.L)) + 1

    normal_x = []
    normal_y = []

    periodic_x = []
    periodic_y = []

    node_x = []
    node_y = []

    for ix in range(ix_min, ix_max + 1):

        for iy in range(iy_min, iy_max + 1):

            shift_x = ix * obj.L
            shift_y = iy * obj.L

            normal_x.append(
                obj.base_normal_x + shift_x
            )

            normal_y.append(
                obj.base_normal_y + shift_y
            )

            periodic_x.append(
                obj.base_periodic_x + shift_x
            )

            periodic_y.append(
                obj.base_periodic_y + shift_y
            )

            node_x.append(
                obj.nodes[:, 0] + shift_x
            )

            node_y.append(
                obj.nodes[:, 1] + shift_y
            )

    normal_x = np.concatenate(normal_x)
    normal_y = np.concatenate(normal_y)

    periodic_x = np.concatenate(periodic_x)
    periodic_y = np.concatenate(periodic_y)

    node_x = np.concatenate(node_x)
    node_y = np.concatenate(node_y)

    box_x = []
    box_y = []

    for ix in range(ix_min, ix_max + 1):

        for iy in range(iy_min, iy_max + 1):

            x0 = ix * obj.L
            y0 = iy * obj.L

            # horizontal
            box_x.extend([
                x0, x0 + obj.L,
                None,
                x0, x0 + obj.L,
                None
            ])

            box_y.extend([
                y0, y0,
                None,
                y0 + obj.L, y0 + obj.L,
                None
            ])

            # vertical
            box_x.extend([
                x0, x0,
                None,
                x0 + obj.L, x0 + obj.L,
                None
            ])

            box_y.extend([
                y0, y0 + obj.L,
                None,
                y0, y0 + obj.L,
                None
            ])
    # Plotly tracesを更新
    with fig.batch_update():

        traces["box"].x = box_x
        traces["box"].y = box_y

        traces["normal"].x = normal_x
        traces["normal"].y = normal_y

        traces["periodic"].x = periodic_x
        traces["periodic"].y = periodic_y

        traces["nodes"].x = node_x
        traces["nodes"].y = node_y

def show_periodic_graph(obj):

    # 基本グラフの幾何情報を作成
    (
        obj.base_normal_x,
        obj.base_normal_y,
        obj.base_periodic_x,
        obj.base_periodic_y
    ) = build_base_graph(obj)

    # FigureWidget
    fig = go.FigureWidget()

    def get_style(name, key, default):
        return obj.styles.get(name, {}).get(key, default)

    def to_color(value):
        if isinstance(value, (tuple, list)):
            return "rgb({}, {}, {})".format(*value)
        return value

    edge_color = to_color(
        get_style("edge", "color", "rgb(90, 38, 32)")
    )
    periodic_color = to_color(
        get_style("edge_periodic", "color", edge_color)
    )
    node_color = to_color(
        get_style("node", "color", edge_color)
    )
    box_color = to_color(
        get_style("box", "color", "gray")
    )

    # 描画trace
    fig.add_trace(go.Scatter(
        x=[], y=[],
        mode="lines",
        line=dict(
            color=box_color,
            width=get_style("box", "width", 1.5),
            dash="dash"
        ),
        hoverinfo="skip",
        showlegend=False
    ))

    fig.add_trace(go.Scatter(
        x=[], y=[],
        mode="lines",
        line=dict(
            color=edge_color,
            width=get_style("edge", "width", 3)
        ),
        hoverinfo="none",
        showlegend=False
    ))

    fig.add_trace(go.Scatter(
        x=[], y=[],
        mode="lines",
        line=dict(
            color=periodic_color,
            width=get_style("edge_periodic", "width", 2)
        ),
        hoverinfo="none",
        showlegend=False
    ))

    fig.add_trace(go.Scatter(
        x=[], y=[],
        mode="lines",
        line=dict(color="rgb(237, 139, 135)", width=8),
        hoverinfo="skip",
        showlegend=False
    ))

    fig.add_trace(go.Scatter(
        x=[], y=[],
        mode="markers",
        marker=dict(
            size=get_style("node", "size", 8),
            color=node_color
        ),
        hoverinfo="skip",
        showlegend=False
    ))

    fig.add_trace(go.Scattergl(
        x=[], y=[],
        mode="markers",
        marker=dict(
            size=1,
            color="rgba(0, 0, 1, 0.001)"
        ),
        hoverinfo="none",
        showlegend=False
    ))

    fig.add_trace(go.Scatter(
        x=[], y=[],
        mode="lines",
        line=dict(color="rgb(96, 124, 131)", width=5),
        hoverinfo="skip",
        showlegend=False
    ))

    fig.add_trace(go.Scatter(
        x=[],
        y=[],
        mode="markers",
        marker=dict(
            color="blue",
            size=12,
            symbol="x"
        ),
        hoverinfo="skip",
        showlegend=False
    ))

    traces = {
        "box": fig.data[0],
        "normal": fig.data[1],
        "periodic": fig.data[2],
        "selected": fig.data[3],
        "nodes": fig.data[4],
        "cursor": fig.data[5],
        "hover": fig.data[6],
        "debug": fig.data[7]
    }

    # 初期表示範囲
    initial_min = -obj.L * obj.fig_frame
    initial_max = obj.L * (1 + obj.fig_frame)

    fig.update_layout(
        width=None,
        height=None,
        autosize=True,
        margin=dict(l=0, r=0, t=0, b=0),
        paper_bgcolor="rgb(248, 246, 235)",
        plot_bgcolor="rgb(248, 246, 235)",
        showlegend=False,
        dragmode="pan",
        hovermode="closest",
        hoverdistance=-1,
        spikedistance=-1,
        xaxis=dict(
            visible=True,
            showticklabels=False,
            showline=False,
            showgrid=False,
            zeroline=False,
            ticks="",
            showspikes=False,
            spikesnap="cursor",
            spikemode="across",
            spikecolor="blue",
            spikethickness=1,
            range=[initial_min, initial_max],
            constrain="domain"
        ),
        yaxis=dict(
            visible=True,
            showticklabels=False,
            showline=False,
            showgrid=False,
            zeroline=False,
            ticks="",
            showspikes=False,
            spikesnap="cursor",
            spikemode="across",
            spikecolor="blue",
            spikethickness=1,
            range=[initial_min, initial_max],
            scaleanchor="x",
            scaleratio=1,
            constrain="domain"
        ),
        uirevision="periodic-graph"
    )

    # 表示範囲の取得
    def get_view_range():
        xr = fig.layout.xaxis.range
        yr = fig.layout.yaxis.range
        return (
            (float(xr[0]), float(xr[1])),
            (float(yr[0]), float(yr[1]))
        )

    def update_cursor_grid():
        (xmin, xmax), (ymin, ymax) = get_view_range()
        
        gx = np.linspace(xmin, xmax, 301)
        gy = np.linspace(ymin, ymax, 301)
        
        xx, yy = np.meshgrid(gx, gy)
        
        traces["cursor"].x = xx.ravel()
        traces["cursor"].y = yy.ravel()

    fig.layout._props["hoveranywhere"] = True # マウスポイントの取得

    # 選択辺の表示
    def set_selected_edge(p1, p2):
        traces["selected"].x = [p1[0], p2[0]]
        traces["selected"].y = [p1[1], p2[1]]

    def clear_selected_edge():
        traces["selected"].x = []
        traces["selected"].y = []

    # ホバー表示
    def on_hover(trace, points, state):
        #print("on_hover called:", points.xs, points.ys)
        if not points.xs:
            return

        x = float(points.xs[0])
        y = float(points.ys[0])

        traces["debug"].x = [x]
        traces["debug"].y = [y]

        result = obj.find_nearest_edge(
            x, y, get_view_range()
        )

        if result is None:
            return

        edge_id, p1, p2 = result

        obj.hover_edge = edge_id
        obj.hover_edge_geometry = (p1.copy(), p2.copy())

        if edge_id == obj.selected_edge:
            clear_selected_edge()
        elif (
            obj.selected_edge is not None
            and obj.selected_edge_geometry is not None
        ):
            p1s, p2s = obj.selected_edge_geometry
            set_selected_edge(p1s, p2s)

        traces["hover"].x = [p1[0], p2[0]]
        traces["hover"].y = [p1[1], p2[1]]

    def on_unhover(trace, points, state):
        obj.hover_edge = None
        obj.hover_edge_geometry = None
        traces["hover"].x = []
        traces["hover"].y = []

        if (
            obj.selected_edge is not None
            and obj.selected_edge_geometry is not None
        ):
            p1, p2 = obj.selected_edge_geometry
            set_selected_edge(p1, p2)

    # クリックによる辺の選択
    def on_click(trace, points, state):
        #print("on_click called:", points.xs, points.ys)
        if not points.xs:
            return

        x = float(points.xs[0])
        y = float(points.ys[0])

        result = obj.find_nearest_edge(
            x, y, get_view_range()
        )

        if result is None:
            return

        edge_id, p1, p2 = result

        if obj.selected_edge == edge_id:
            obj.selected_edge = None
            obj.selected_edge_geometry = None
            clear_selected_edge()
            return

        obj.selected_edge = edge_id
        obj.selected_edge_geometry = (
            p1.copy(), p2.copy()
        )

        set_selected_edge(p1, p2)

        obj.hover_edge = None
        obj.hover_edge_geometry = None
        traces["hover"].x = []
        traces["hover"].y = []

    # 辺traceにイベントを接続
    for trace in (traces["normal"], traces["periodic"], traces["cursor"]):
        trace.on_hover(on_hover)
        trace.on_unhover(on_unhover)
        trace.on_click(on_click)

    # ズーム・パン時の再描画
    def on_range_change(layout, x_range, y_range):
        update_view(obj, fig, traces)
        update_cursor_grid()

    fig.layout.on_change(
        on_range_change,
        "xaxis.range",
        "yaxis.range"
    )

    # 初期描画
    update_view(obj, fig, traces)

    for name, tr in traces.items():
        x = np.asarray([v for v in tr.x if v is not None], dtype=float)
        y = np.asarray([v for v in tr.y if v is not None], dtype=float)
        
        if len(x) == 0 or len(y) == 0:
            continue
         
    # UI
    score_widget = widgets.HTML()
    message_widget = widgets.HTML()

    update_score(
        score_widget,
        obj.best_score
    )

    display_message(
        message_widget,
        "Ready",
        obj.best_score,
        obj.best_score_step
    )

    # 履歴操作
    step_spinbox = widgets.BoundedIntText(
        value=obj.current_step,
        min=0,
        max=len(obj.move_history),
        layout=widgets.Layout(width="75px")
    )

    max_step_label = widgets.HTML(
        value=f"/{len(obj.move_history)}"
    )

    # JSON保存・読み込み
    download = widgets.HTML()

    def update_download_link():
        data = graph_to_json(obj)
        encoded = base64.b64encode(
            data.encode("utf-8")
        ).decode("ascii")

        download.value = (
            '<a download="periodic_graph.json" '
            'href="data:application/json;base64,'
            + encoded
            + '">Download JSON</a>'
        )

    update_download_link()

    upload = widgets.FileUpload(
        accept=".json",
        multiple=False,
        description="Upload JSON"
    )

    # 操作後の画面更新
    def refresh(result=None, default_message="Updated"):
        (
            obj.base_normal_x,
            obj.base_normal_y,
            obj.base_periodic_x,
            obj.base_periodic_y
        ) = build_base_graph(obj)

        update_view(obj, fig, traces)
        update_cursor_grid()

        update_history_controls(
            obj,
            step_spinbox,
            max_step_label
        )

        if result is None:
            message = default_message
        else:
            message = result.get("message", default_message)

        #entropy_shannon, _ = obj.calculate_cycle_entropy()
        #score = 1e6 * entropy_shannon

        #update_score(score_widget, score)
        
        if result is not None and "score" in result:
            update_score(score_widget, result["score"])
        elif result is None:
            entropy_shannon, _ = obj.calculate_cycle_entropy()
            update_score(score_widget, 1e6 * entropy_shannon)

        display_message(
            message_widget,
            message,
            obj.best_score,
            obj.best_score_step
        )

        update_download_link()

    # ボタン操作
    def flip_action(button):
        result = obj.flip_selected_edge()
        if result is None:
            refresh(default_message="No edge selected")
        else:
            refresh(result)

    def random_flip_action(button):
        result = obj.random_flip_edge()
        if result is None:
            refresh(default_message="No edge available")
        else:
            refresh(result)

    def relax_action(button):
        result = obj.relax_structure(
            time_steps=100,
            k_spring=0.1
        )
        refresh(result, "Relaxed")

    def undo_action(button):
        result = obj.undo()
        if result is not None:
            refresh(result)

    def redo_action(button):
        result = obj.redo()
        if result is not None:
            refresh(result)

    def go_action(button):
        result = obj.go_to_step(step_spinbox.value)
        if result is not None:
            refresh(result)

    flip_button = widgets.Button(
        description="Diagonal Flip ↻",
        button_style=""
    )
    flip_button.on_click(flip_action)

    random_flip_button = widgets.Button(
        description="Random Diagonal Flip ↻"
    )
    random_flip_button.on_click(random_flip_action)

    relax_button = widgets.Button(
        description="Relax"
    )
    relax_button.on_click(relax_action)

    undo_button = widgets.Button(
        description="Step Back"
    )
    undo_button.on_click(undo_action)

    redo_button = widgets.Button(
        description="Next Step"
    )
    redo_button.on_click(redo_action)

    go_button = widgets.Button(
        description="Go"
    )
    go_button.on_click(go_action)

    # JSONアップロードイベント
    def on_upload(change):
        value = change["new"]

        if not value:
            return

        if isinstance(value, dict):
            file_info = next(iter(value.values()))
        else:
            file_info = value[0]

        content = file_info["content"]

        if isinstance(content, memoryview):
            content = content.tobytes()

        data = json.loads(
            bytes(content).decode("utf-8")
        )

        result = load_graph_data(obj, data)
        refresh(result, "Loaded JSON")

    upload.observe(on_upload, names="value")

    # UI配置
    button_row = widgets.HBox([
        flip_button,
        random_flip_button,
        relax_button
    ])

    history_row = widgets.HBox([
        undo_button,
        redo_button,
        step_spinbox,
        max_step_label,
        go_button
    ])

    file_row = widgets.HBox([
        download,
        upload
    ])

    app = widgets.VBox([
        score_widget,
        fig,
        message_widget,
        button_row,
        history_row,
        file_row
    ])

    #display(app)

    # 後から操作できるように参照を保持
    obj.fig = fig
    obj.traces = traces
    obj.app = app
    obj.score_widget = score_widget
    obj.message_widget = message_widget
    obj.step_spinbox = step_spinbox
    obj.max_step_label = max_step_label
    obj.upload = upload
    obj.download = download

    return app

class Periodic:
    def __init__(self, L, fig_frame, plot_styles=None):
        self.L = L
        self.fig_frame = fig_frame

        self.shifts = np.array([
            [-L, -L],
            [-L,  0],
            [-L,  L],
            [ 0, -L],
            [ 0,  0],
            [ 0,  L],
            [ L, -L],
            [ L,  0],
            [ L,  L]
        ])

        default_styles = {
            "edge": {
                "color": (90, 38, 32),
                "width": 6
            },
            "edge_periodic": {
                "color": (90, 38, 32),
                "width": 3
            },
            "node": {
                "size": 12,
                "color": (90, 38, 32)
            },
            "box": {
                "color": (128, 128, 128),
                "width": 1.5
            }
        }

        self.styles = default_styles

        if plot_styles is not None:
            for key, value in plot_styles.items():
                if key in self.styles and isinstance(value, dict):
                    self.styles[key].update(value)

        self.selected_edge = None
        self.selected_edge_geometry = None
        self.hover_edge = None
        self.hover_edge_geometry = None

        self.best_score = None
        self.best_score_step = None

        self.calculate_km_reference()
        
    def score_km_wasserstein(self):
        distance = self.calculate_spectral_distance()
        
        if self.initial_distance == 0:
            return 0.0
        
        score = distance / self.initial_distance
        #score = (score - 0.0)/(1 - 0.0)
        score = - 100 * np.log10(score)
        
        return max(score, 1e-12)

    def load_json(self, fname, magnify):
        self.fname = fname
        with open(self.fname + ".json", "r") as f:
            data = json.load(f)

        try:
            H = json_graph.node_link_graph(data)
        except:
            H = json_graph.node_link_graph(data, edges="links")
        nodes_load = H.nodes(data=True)
        edges_load = H.edges()

        self.nodes = magnify * np.array([
            [value["x"], value["y"]] for _, value in nodes_load
        ]) % self.L

        self.edges = [(int(u), int(v)) for u, v in edges_load]

        self.initial_edges = self.edges.copy()
        self.move_history = []
        self.current_step = 0

        #self.initial_distance = self.calculate_spectral_distance()
        entropy_shannon, cycle_counts = self.calculate_cycle_entropy()

        self.best_score = 1e6 * entropy_shannon
        self.best_score_step = self.current_step

    def calculate_km_reference(self, n_grid=10000):
        """Kesten–McKay distribution for 3-regular graphs."""
        x = np.linspace(-2 * np.sqrt(2), 2 * np.sqrt(2), n_grid)
        rho = (
            3 * np.sqrt(np.maximum(8 - x**2, 0))
            / (2 * np.pi * (9 - x**2))
        )
        
        dx = x[1] - x[0]
        
        cdf = np.zeros_like(x)
        cdf[1:] = np.cumsum(
            0.5 * (rho[:-1] + rho[1:]) * dx
        )
        
        cdf /= cdf[-1]
        
        self.km_x = x
        self.km_cdf = cdf
    
    def set_reference_spectrum(self):
        """現在のグラフのスペクトルを基準として保存する"""
        self.reference_spectrum = self.calculate_laplacian_eigenvalues()

    def calculate_von_neumann_entropy(self):
        N = len(self.nodes)
        A = np.zeros((N, N), dtype=float)
        
        for u, v in self.edges:
            A[u, v] = 1.0
            A[v, u] = 1.0
        
        L = 3.0 * np.eye(N) - A
        
        eigenvalues = np.linalg.eigvalsh(L)
        eigenvalues = np.clip(eigenvalues, 0.0, None)
        
        p = eigenvalues / eigenvalues.sum()
        p = p[p > 0]
        
        entropy = -np.sum(p * np.log(p))
        
        return entropy

    def calculate_laplacian_eigenvalues(self):
        N = len(self.nodes)
        A = np.zeros((N, N), dtype=float)
        
        for u, v in self.edges:
            A[u, v] = 1.0
            A[v, u] = 1.0
        
        L = 3.0 * np.eye(N) - A
        
        return np.linalg.eigvalsh(L)

    def calculate_spectral_distance(self):
        """Wasserstein-1 distance from the Kesten–McKay distribution."""
        N = len(self.nodes)
        A = np.zeros((N, N), dtype=float)
        
        for u, v in self.edges:
            A[u, v] = 1.0
            A[v, u] = 1.0
        
        eigenvalues = np.linalg.eigvalsh(A)
        
        p = (np.arange(N) + 0.5) / N # Empirical spectral quantiles
        km_quantiles = np.interp(p, self.km_cdf, self.km_x) # Kesten–McKay quantiles
        
        return np.mean(np.abs(
            np.sort(eigenvalues) - km_quantiles
        ))

    def calculate_cycle_entropy(self, max_len=24):
        from collections import deque
        N = len(self.nodes)

        adj = [set() for _ in range(N)]
        for u, v in self.edges:
            if u != v:
                adj[u].add(v)
                adj[v].add(u)

        cycle_lengths = []


        for node in range(N):
            neighbors = sorted(adj[node])
            min_cycle = None

            # Examine every pair of neighbours
            for i, a in enumerate(neighbors):
                for b in neighbors[i + 1:]:

                    # BFS from a to b, excluding the original node
                    queue = deque([(a, 0)])
                    visited = {a}

                    while queue:
                        current, distance = queue.popleft()

                        # Close the cycle
                        if current == b:
                            cycle_length = distance + 2

                            if min_cycle is None or cycle_length < min_cycle:
                                min_cycle = cycle_length

                            break

                        # Maximum cycle length is max_len
                        if distance >= max_len - 2:
                            continue

                        for nxt in adj[current]:
                            if nxt == node or nxt in visited:
                                continue

                            visited.add(nxt)
                            queue.append((nxt, distance + 1))

            cycle_lengths.append(min_cycle)

        # Count the minimum cycle lengths
        cycle_counts = {}
        for length in cycle_lengths:
            if length is not None:
                cycle_counts[length] = cycle_counts.get(length, 0) + 1

        # Calculate Shannon entropy
        total = sum(cycle_counts.values())

        if total == 0:
            entropy = 0.0
        else:
            p = np.array(list(cycle_counts.values()), dtype=float) / total
            entropy = -np.sum(p * np.log(p))

        unclassified = cycle_lengths.count(None)

        return entropy, dict(sorted(cycle_counts.items()))
    
    def point_to_segment_distance(self, point, p1, p2):
        v = p2 - p1
        vv = np.dot(v, v)
        
        if vv == 0:
            return np.linalg.norm(point - p1)
        
        t = np.dot(point - p1, v) / vv
        t = np.clip(t, 0.01, 0.99)
        
        closest = p1 + t * v
        
        return np.linalg.norm(point - closest)

    def find_nearest_edge(self, x, y, view_range):
        point = np.array([x, y])

        ix = int(np.floor(x / self.L))
        iy = int(np.floor(y / self.L))

        best_distance = np.inf
        best_edge_id = None
        best_p1 = None
        best_p2 = None

        for dx in (-1, 0, 1):
            for dy in (-1, 0, 1):
                shift = np.array([
                    (ix + dx) * self.L,
                    (iy + dy) * self.L
                ])

                for geometry in self.edge_geometry:
                    p1 = geometry["p1"] + shift
                    p2 = geometry["p2"] + shift

                    distance = self.point_to_segment_distance(
                        point,
                        p1,
                        p2
                    )

                    if distance < best_distance:
                        best_distance = distance
                        best_edge_id = geometry["edge_id"]
                        best_p1 = p1.copy()
                        best_p2 = p2.copy()

        xrange = view_range[0][1] - view_range[0][0]
        yrange = view_range[1][1] - view_range[1][0]

        threshold = max(xrange, yrange) * 0.015

        if best_distance <= threshold:
            return (
                best_edge_id,
                best_p1,
                best_p2
            )

        return None

    def flip_selected_edge(self):
        if self.selected_edge is None:
            return {
                "success": False,
                "message": "No edge is selected."
            }

        edge_id = self.selected_edge
        nodes_before = self.nodes.copy()

        a, b = self.edges[edge_id]

        p_a = self.nodes[a]
        p_b = self.nodes[b]

        disp_ab, _ = minimum_image_displacement(
            self,
            p_a,
            p_b
        )

        p_b_image = p_a + disp_ab

        vector = p_b_image - p_a
        center = 0.5 * (p_a + p_b_image)

        a_neighbors = []
        b_neighbors = []

        for eid, (u, v) in enumerate(self.edges):
            if eid == edge_id:
                continue

            if u == a:
                a_neighbors.append((eid, v))
            elif v == a:
                a_neighbors.append((eid, u))

            if u == b:
                b_neighbors.append((eid, v))
            elif v == b:
                b_neighbors.append((eid, u))

        combined_neighbors = (
            [neighbor[1] for neighbor in a_neighbors]
            + [neighbor[1] for neighbor in b_neighbors]
        )

        if len(combined_neighbors) != len(set(combined_neighbors)):
            message = (
                "The selected edge that constitutes a triangle "
                "cannot be flipped: "
                f"{combined_neighbors}\n"
                "Consider flipping another edge."
            )

            self.selected_edge = None
            self.selected_edge_geometry = None

            return {
                "success": False,
                "message": message
            }

        if len(a_neighbors) != 2:
            return {
                "success": False,
                "message": (
                    f"Error: node {a} has "
                    f"{len(a_neighbors)} other edges."
                )
            }

        if len(b_neighbors) != 2:
            return {
                "success": False,
                "message": (
                    f"Error: node {b} has "
                    f"{len(b_neighbors)} other edges."
                )
            }

        surrounding = []

        for endpoint, neighbours, sign in zip(
            ["a", "b"],
            [a_neighbors, b_neighbors],
            [-1, +1]
        ):
            for eid, node in neighbours:
                p = self.nodes[node]

                disp, _ = minimum_image_displacement(
                    self,
                    center,
                    p
                )

                position = center + disp

                angle = arg_vec(
                    sign * vector,
                    position - center
                )

                surrounding.append({
                    "edge_id": eid,
                    "node": node,
                    "endpoint": endpoint,
                    "position": position,
                    "angle": angle
                })

        surrounding.sort(
            key=lambda x: x["angle"],
            reverse=True
        )

        a_items = [
            item
            for item in surrounding
            if item["endpoint"] == "a"
        ]

        b_items = [
            item
            for item in surrounding
            if item["endpoint"] == "b"
        ]

        a1 = a_items[0]
        a2 = a_items[1]

        b1 = b_items[0]
        b2 = b_items[1]

        R = np.array([
            [0, -1],
            [+1, 0]
        ])

        self.nodes[a] = (
            center + 0.5 * R @ vector
        ) % self.L

        self.nodes[b] = (
            center - 0.5 * R @ vector
        ) % self.L

        removed_edges = [
            self.edges[a1["edge_id"]],
            self.edges[b1["edge_id"]]
        ]

        new_edges = [
            (a1["node"], b),
            (b1["node"], a)
        ]

        move_record = {
            "removed_edges": removed_edges,
            "added_edges": new_edges.copy(),
            "nodes_before": nodes_before.tolist(),
            "nodes_after": self.nodes.tolist()
        }

        for eid, edge in zip(
            [a1["edge_id"], b1["edge_id"]],
            new_edges
        ):
            self.edges[eid] = edge

        self.move_history = self.move_history[
            :self.current_step
        ]

        self.move_history.append(move_record)
        self.current_step += 1

        self.selected_edge = None
        self.selected_edge_geometry = None

        self.hover_edge = None
        self.hover_edge_geometry = None

        #entropy = self.calculate_von_neumann_entropy()

        entropy_shannon, cycle_counts = (
            self.calculate_cycle_entropy()
        )

        score = 1e6 * entropy_shannon

        if score > self.best_score:
            self.best_score = score
            self.best_score_step = self.current_step

        message = "Successfully flipped!"
        message += (
            "\n"
            #+ f"von Neumann entropy: {entropy}"
            + f", n-gon Shannon entropy: {entropy_shannon}"
        )

        return {
            "success": True,
            "message": message,
            "score": score,
            #"entropy": entropy,
            "cycle_counts": cycle_counts,
            "step": self.current_step
        }

    def random_flip_edge(self):
        if not self.edges:
            return None

        edge_id = random.choice(range(len(self.edges)))

        self.selected_edge = edge_id
        return self.flip_selected_edge()

    def undo(self):
        if self.current_step > 0:
            return self.recover_graph(
                self.current_step - 1,
                force_current=True
            )
        return None

    def redo(self):
        if self.current_step < len(self.move_history):
            return self.recover_graph(
                self.current_step + 1,
                force_current=True
            )
        return None

    def go_to_step(self, step):
        if 0 <= step <= len(self.move_history):
            return self.recover_graph(
                step,
                force_current=False
            )
        return None
        
    def apply_move(self, edges, move, forward=True):
        before = (
            move["removed_edges"]
            if forward
            else move["added_edges"]
        )

        after = (
            move["added_edges"]
            if forward
            else move["removed_edges"]
        )

        for old_edge, new_edge in zip(before, after):
            key = frozenset(old_edge)

            for eid, edge in enumerate(edges):
                if frozenset(edge) == key:
                    edges[eid] = tuple(new_edge)
                    break
            else:
                raise ValueError(
                    f"Edge {old_edge} not found in graph."
                )

        return edges

    def recover_graph(self, target_step, force_current=False):
        if not 0 <= target_step <= len(self.move_history):
            return {
                "success": False,
                "message": f"Invalid step: {target_step}"
            }

        edges = self.edges.copy()
        nodes = self.nodes.copy()

        if target_step > self.current_step:
            for i in range(self.current_step, target_step):
                move = self.move_history[i]

                edges = self.apply_move(
                    edges,
                    move,
                    forward=True
                )

                if "nodes_after" in move:
                    nodes = np.array(move["nodes_after"])

        elif target_step < self.current_step:
            for i in range(self.current_step - 1, target_step - 1, -1):
                move = self.move_history[i]

                edges = self.apply_move(
                    edges,
                    move,
                    forward=False
                )

                if "nodes_before" in move:
                    nodes = np.array(move["nodes_before"])

        self.edges = edges
        self.nodes = nodes
        self.current_step = target_step

        self.selected_edge = None
        self.selected_edge_geometry = None
        self.hover_edge = None
        self.hover_edge_geometry = None

        #entropy = self.calculate_von_neumann_entropy()
        entropy_shannon, cycle_counts = (
            self.calculate_cycle_entropy()
        )

        score = 1e6 * entropy_shannon

        if score > self.best_score:
            self.best_score = score
            self.best_score_step = self.current_step

        message = (
            f"Recovered graph at step {self.current_step}\n"
            #f"von Neumann entropy: {entropy}, "
            f"n-gon Shannon entropy: {entropy_shannon}"
        )

        return {
            "success": True,
            "message": message,
            "score": score,
            #"entropy": entropy,
            "cycle_counts": cycle_counts,
            "step": self.current_step
        }

    def spring_relax(self, time_steps, k_spring=0.01, l_free=None):
        nodes_sim = self.nodes.copy()

        if l_free is None:
            edge_lengths = []

            for i, j in self.edges:
                disp, _ = minimum_image_displacement(
                    self,
                    nodes_sim[i],
                    nodes_sim[j]
                )

                edge_lengths.append(
                    np.linalg.norm(disp)
                )

            l_free = np.mean(edge_lengths) * 0.8

        l_free_sq = l_free ** 2

        forces = np.zeros_like(nodes_sim)

        for time_step in range(time_steps):
            sum_disp = 0.0
            forces.fill(0.0)

            for i, j in self.edges:
                p1 = nodes_sim[i]
                p2 = nodes_sim[j]

                disp, k = minimum_image_displacement(
                    self,
                    p1,
                    p2
                )

                dist_sq = np.sum(disp ** 2)

                pforce1 = -k_spring * (
                    l_free_sq / dist_sq - 1
                ) * disp

                forces[i] += pforce1
                forces[j] -= pforce1

                sum_disp += np.sqrt(dist_sq)

            nodes_sim += forces
            nodes_sim %= self.L

            if (
                time_step % 100 == 0
                or time_step == time_steps - 1
            ):
                mean_length = (
                    sum_disp / len(self.edges)
                )

        return nodes_sim

    
    def relax_structure(
        self,
        time_steps=100,
        k_spring=0.1,
        l_free=None
    ):
        new_nodes = self.spring_relax(
            time_steps=time_steps,
            k_spring=k_spring,
            l_free=l_free
        )

        self.nodes = new_nodes

        (
            self.base_normal_x,
            self.base_normal_y,
            self.base_periodic_x,
            self.base_periodic_y
        ) = build_base_graph(self)

        return {
            "success": True,
            "message": "Relaxed"
        }

display(HTML("""
<style>
.plotly-graph-div {
    width: min(100%, 800px) !important;
    aspect-ratio: 1 / 1;
}
</style>
"""))

fig_frame = 0.1

L = 100
#fname = "graph_20260722_240_360_083719"
#magnify = 4.3
fname = "../graph_20260721_60_90_235627"
magnify = 10

pgraph = Periodic(L, fig_frame)
try:
    pgraph.load_json(fname, magnify)
except:
    pgraph.load_json(fname.replace("../", ""), magnify)

show_periodic_graph(pgraph)